# Combined reporter titration

Low-range titration curves (cells per guide ≤ 210) of distinctiveness and EBI mAP for the combined-reporter aggregates: cp (7 reporters combined) and live-cell-matched (7 reporters combined).

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

FIGURES_DIR = Path("../../output/figure_3")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## CSV paths

Combined-reporter aggregate titration CSV (long form: `group`, `x`, `metric`, `y`).

**Before public release**, replace this cell with download instructions (or a pointer to the public dataset) and update the constant to match the released layout.

In [ ]:
COMBINED_CSV = "/hpc/projects/icd.fast.ops/organelle_attribution/pca_optimized_v0.3/cell_dino/zscore_per_exp/paper_v1/with_cp/with_4i/all_livecell/fixed_80%/cosine/combined_titration_compare/per_guide_median/cp_vs_4i_vs_matched_livecell_best_vs_livecell/compare_all_metrics_cells_per_guide.csv"

## Load and preprocess

Pivot the combined-reporter CSV from long to wide and split it into one dataframe per modality.

In [ ]:
GROUP_REMAP = {
    "cp":                    "cp",
    "4i":                    "4i",
    "matched_livecell_best": "live-cell",
    "livecell":              "livecell",
}

_cdf = pd.read_csv(COMBINED_CSV)
combined = {}
for group, grp in _cdf.groupby("group"):
    pivot = grp.pivot(index="x", columns="metric", values="y").reset_index()
    pivot = pivot.rename(columns={
        "x":               "cells_per_guide",
        "distinctiveness": "distinctiveness_map_mean",
        "ebi":             "ebi_map_mean",
        "activity":        "activity_map_mean",
        "corum":           "corum_map_mean",
    })
    combined[GROUP_REMAP.get(group, group)] = pivot.sort_values("cells_per_guide")

## Shared plot styling

Per-modality color used across both plots.

In [ ]:
COMBINED_COLORS = {"4i": "#f28c3a", "cp": "#7b5ea7", "live-cell": "#555555", "livecell": "#555555"}

## Plot 1 — Distinctiveness, combined-reporter aggregates (low range)

Combined-reporter distinctiveness curves for cp and live-cell-matched at low cells-per-guide.

In [ ]:
COMBINED_LABELS = {
    "live-cell": "live-cell-matched (7 reporters combined)",
    "4i":        "4i (10 reporters combined)",
    "cp":        "cp (7 reporters combined)",
    "livecell":  "live-cell (40 reporters combined)",
}

fig, ax = plt.subplots(figsize=(4, 2))

for modality, df_c in combined.items():
    if modality in ("4i", "livecell"):
        continue
    df_c = df_c.sort_values("cells_per_guide")
    ax.plot(
        df_c["cells_per_guide"],
        df_c["distinctiveness_map_mean"],
        color=COMBINED_COLORS[modality],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label=COMBINED_LABELS[modality],
        zorder=5,
    )

ax.set_xlim(1, 210)
ax.set_ylim(0, 0.2)
ax.set_yticks(np.arange(0, 0.21, 0.1))
ax.set_xlabel("Cells per guide")
ax.set_ylabel("Distinctiveness mAP")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_distinctiveness_combined.svg", bbox_inches="tight")
plt.show()

## Plot 2 — EBI, combined-reporter aggregates (low range)

Same low-range view as plot 1 but for EBI mAP.

In [ ]:
COMBINED_CHAD_LABELS = {
    "live-cell": "live-cell-matched (7 reporters combined)",
    "4i":        "4i (10 reporters combined)",
    "cp":        "cp (7 reporters combined)",
    "livecell":  "live-cell (all reporters combined)",
}

fig, ax = plt.subplots(figsize=(4, 2))

for modality, df_c in combined.items():
    if modality in ("4i", "livecell"):
        continue
    df_c = df_c.sort_values("cells_per_guide")
    ax.plot(
        df_c["cells_per_guide"],
        df_c["ebi_map_mean"],
        color=COMBINED_COLORS[modality],
        linewidth=2.5,
        marker="o",
        markersize=5,
        label=COMBINED_CHAD_LABELS[modality],
        zorder=5,
    )

ax.set_xlim(1, 210)
ax.set_ylim(0, 0.43)
ax.set_xlabel("Cells per guide")
ax.set_ylabel("EBI mAP")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "titration_ebi_combined.svg", bbox_inches="tight")
plt.show()